# Pipeline support functions

Reusable imports, constants, classes, and functions extracted from `301_pipeline_report.ipynb`. Load this notebook from the main notebook with `%run ./301_pipeline_support_functions.ipynb`.

In [ ]:
from pathlib import Path

import pydicom

import numpy as np

import pandas as pd

from pathlib import Path

import os

import pandas as pd

from sklearn.model_selection import GroupShuffleSplit

from dataclasses import dataclass

from pathlib import Path

import numpy as np

import torch

import torch.nn.functional as F

import pydicom

import matplotlib.pyplot as plt

import cv2

from pydicom.pixel_data_handlers.util import apply_voi_lut

In [ ]:
import os

In [ ]:
def infer_plane(desc):
    desc = str(desc).lower()
    if "sag" in desc:
        return "Sagittal"
    if "ax" in desc:
        return "Axial"
    return "Unknown"

In [ ]:
from pathlib import Path

import pandas as pd

In [ ]:
@dataclass
class VNetCFG:
    fixed_depth: int = 32
    img_size: int = 256
    base_channels: int = 8
    levels: tuple = ("L1/L2", "L2/L3", "L3/L4", "L4/L5", "L5/S1")

vnet_cfg = VNetCFG()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
def read_dicom_image(path):
    ds = pydicom.dcmread(str(path))

    if apply_voi_lut is not None:
        try:
            img = apply_voi_lut(ds.pixel_array, ds).astype(np.float32)
        except Exception:
            img = ds.pixel_array.astype(np.float32)
    else:
        img = ds.pixel_array.astype(np.float32)

    if getattr(ds, "PhotometricInterpretation", "") == "MONOCHROME1":
        img = img.max() - img

    return ds, img

def normalize_image(img):
    lo, hi = np.percentile(img, [1, 99])
    img = np.clip(img, lo, hi)
    img = (img - lo) / (hi - lo + 1e-6)
    return img.astype(np.float32)

def resize_image(img, size):
    if img.shape == (size, size):
        return img.astype(np.float32)

    if cv2 is not None:
        return cv2.resize(img, (size, size), interpolation=cv2.INTER_AREA).astype(np.float32)

    x = torch.from_numpy(img)[None, None].float()
    x = F.interpolate(x, size=(size, size), mode="bilinear", align_corners=False)
    return x[0, 0].numpy().astype(np.float32)

In [ ]:
def load_series_volume(series_df, cfg=vnet_cfg):
    series_df = series_df.sort_values("slice_number").reset_index(drop=True)

    slices = []
    original_shapes = []

    for path in series_df["img_path"]:
        ds, img = read_dicom_image(path)

        original_shapes.append((int(ds.Rows), int(ds.Columns)))

        img = normalize_image(img)
        img = resize_image(img, cfg.img_size)

        slices.append(img)

    vol = np.stack(slices).astype(np.float32)  # [D, H, W]
    original_depth = vol.shape[0]

    x = torch.from_numpy(vol)[None, None]  # [1, 1, D, H, W]
    x = F.interpolate(
        x,
        size=(cfg.fixed_depth, cfg.img_size, cfg.img_size),
        mode="trilinear",
        align_corners=False,
    )

    vol = x[0, 0].numpy().astype(np.float32)

    vol = (vol - vol.mean()) / (vol.std() + 1e-6)

    meta = {
        "original_depth": original_depth,
        "original_shapes": original_shapes,
    }

    return vol, meta

In [ ]:
import torch

import torch.nn as nn

import torch.nn.functional as F

class ResidualConvBlock3D(nn.Module):
    def __init__(self, in_ch, out_ch, n_convs=2, kernel_size=3, dropout=0.0):
        super().__init__()

        padding = kernel_size // 2
        layers = []
        ch = in_ch

        for _ in range(n_convs):
            layers += [
                nn.Conv3d(ch, out_ch, kernel_size=kernel_size, padding=padding, bias=False),
                nn.InstanceNorm3d(out_ch, affine=True),
                nn.PReLU(out_ch),
            ]

            if dropout > 0:
                layers.append(nn.Dropout3d(dropout))

            ch = out_ch

        self.net = nn.Sequential(*layers)
        self.proj = (
            nn.Identity()
            if in_ch == out_ch
            else nn.Conv3d(in_ch, out_ch, kernel_size=1, bias=False)
        )
        self.act = nn.PReLU(out_ch)

    def forward(self, x):
        return self.act(self.net(x) + self.proj(x))

class DownBlock3D(nn.Module):
    def __init__(self, in_ch, out_ch, n_convs=2):
        super().__init__()

        self.down = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, kernel_size=2, stride=2, bias=False),
            nn.InstanceNorm3d(out_ch, affine=True),
            nn.PReLU(out_ch),
        )

        self.block = ResidualConvBlock3D(out_ch, out_ch, n_convs=n_convs)

    def forward(self, x):
        return self.block(self.down(x))

class UpBlock3D(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch, n_convs=2):
        super().__init__()

        self.up = nn.Sequential(
            nn.ConvTranspose3d(in_ch, out_ch, kernel_size=2, stride=2, bias=False),
            nn.InstanceNorm3d(out_ch, affine=True),
            nn.PReLU(out_ch),
        )

        self.skip_proj = (
            nn.Identity()
            if skip_ch == out_ch
            else nn.Conv3d(skip_ch, out_ch, kernel_size=1, bias=False)
        )

        self.block = ResidualConvBlock3D(out_ch, out_ch, n_convs=n_convs)

    def forward(self, x, skip):
        x = self.up(x)

        if x.shape[-3:] != skip.shape[-3:]:
            x = F.interpolate(
                x,
                size=skip.shape[-3:],
                mode="trilinear",
                align_corners=False,
            )

        x = x + self.skip_proj(skip)
        return self.block(x)

class VNet3DLocalization(nn.Module):
    def __init__(self, in_channels=1, out_channels=5, base_channels=8):
        super().__init__()

        c = base_channels

        self.enc1 = ResidualConvBlock3D(in_channels, c, n_convs=1)
        self.enc2 = DownBlock3D(c, c * 2, n_convs=2)
        self.enc3 = DownBlock3D(c * 2, c * 4, n_convs=2)
        self.enc4 = DownBlock3D(c * 4, c * 8, n_convs=2)

        self.bottleneck = DownBlock3D(c * 8, c * 16, n_convs=2)

        self.dec4 = UpBlock3D(c * 16, c * 8, c * 8, n_convs=2)
        self.dec3 = UpBlock3D(c * 8, c * 4, c * 4, n_convs=2)
        self.dec2 = UpBlock3D(c * 4, c * 2, c * 2, n_convs=2)
        self.dec1 = UpBlock3D(c * 2, c, c, n_convs=1)

        self.out = nn.Conv3d(c, out_channels, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)

        b = self.bottleneck(e4)

        d4 = self.dec4(b, e4)
        d3 = self.dec3(d4, e3)
        d2 = self.dec2(d3, e2)
        d1 = self.dec1(d2, e1)

        return self.out(d1)

In [ ]:
def load_vnet_checkpoint(checkpoint_path, cfg=vnet_cfg):
    model = VNet3DLocalization(
        in_channels=1,
        out_channels=len(cfg.levels),
        base_channels=cfg.base_channels,
    ).to(device)

    ckpt = torch.load(checkpoint_path, map_location=device)

    if "model_state_dict" in ckpt:
        state_dict = ckpt["model_state_dict"]
    else:
        state_dict = ckpt

    model.load_state_dict(state_dict)
    model.eval()

    return model

In [ ]:
@torch.no_grad()
def predict_vnet_levels(model, vol, meta, cfg=vnet_cfg):
    x = torch.from_numpy(vol)[None, None].float().to(device)

    logits = model(x)
    probs = torch.sigmoid(logits)[0].cpu().numpy()  # [5, D, H, W]

    # collapse depth and width, keep only y/height profile
    y_profiles = probs.max(axis=(1, 3))  # [5, H]

    pred_y_resized = y_profiles.argmax(axis=1)
    confidence = y_profiles.max(axis=1)

    h_original = meta["original_shapes"][0][0]

    pred_y_original = pred_y_resized / (cfg.img_size - 1) * (h_original - 1)

    pred_df = pd.DataFrame({
        "level": list(cfg.levels),
        "level_idx": range(len(cfg.levels)),
        "pred_y_resized": pred_y_resized,
        "pred_y_original": pred_y_original,
        "confidence": confidence,
    })

    return pred_df, probs, y_profiles

In [ ]:
def get_slice_geometry(path):
    ds = pydicom.dcmread(str(path), stop_before_pixels=True)

    ipp = np.array(ds.ImagePositionPatient, dtype=float)
    iop = np.array(ds.ImageOrientationPatient, dtype=float)

    row_dir = iop[:3]
    col_dir = iop[3:]
    normal = np.cross(row_dir, col_dir)

    return {
        "img_path": str(path),
        "ipp_x": ipp[0],
        "ipp_y": ipp[1],
        "ipp_z": ipp[2],
        "row_x": row_dir[0],
        "row_y": row_dir[1],
        "row_z": row_dir[2],
        "col_x": col_dir[0],
        "col_y": col_dir[1],
        "col_z": col_dir[2],
        "normal_x": normal[0],
        "normal_y": normal[1],
        "normal_z": normal[2],
        "pixel_spacing_0": float(ds.PixelSpacing[0]),
        "pixel_spacing_1": float(ds.PixelSpacing[1]),
        "rows": int(ds.Rows),
        "cols": int(ds.Columns),
    }

In [ ]:
LEVEL_ORDER = ["L1/L2", "L2/L3", "L3/L4", "L4/L5", "L5/S1"]

RELATIVE_SLICE_CENTER_RULES = {
    "spinal_canal": {"all": 0.50},
    "left_foraminal": {
        "all": 0.29,
        "L1/L2": 0.30,
        "L2/L3": 0.30,
        "L3/L4": 0.29,
        "L4/L5": 0.28,
        "L5/S1": 0.27,
    },
    "right_foraminal": {
        "all": 0.76,
        "L1/L2": 0.73,
        "L2/L3": 0.74,
        "L3/L4": 0.76,
        "L4/L5": 0.76,
        "L5/S1": 0.78,
    },
}

def normalize_condition_for_slice_selection(row):
    text = f"{row.get('condition', '')} {row.get('row_id', '')}".lower()

    if "spinal canal stenosis" in text:
        return "spinal_canal"

    if "foraminal" in text:
        if "left" in text:
            return "left_foraminal"
        if "right" in text:
            return "right_foraminal"

    return None

def get_level_key(row):
    level = row.get("level", None)
    return str(level) if str(level) in LEVEL_ORDER else "all"

def rel_to_rank(rel_pos, n_slices):
    if n_slices <= 1:
        return 1

    rel_pos = float(np.clip(rel_pos, 0, 1))
    return int(round(rel_pos * (n_slices - 1))) + 1

In [ ]:
def select_sagittal_slices(study_df, study_geom):
    sagittal_geom = study_geom[study_geom["plane"].eq("Sagittal")].copy()

    selected = []

    for _, row in study_df.iterrows():
        task = normalize_condition_for_slice_selection(row)

        if task is None:
            continue

        series_id = str(row["series_id"])
        level = get_level_key(row)

        series_slices = sagittal_geom[
            sagittal_geom["series_id"].astype(str).eq(series_id)
        ].sort_values("slice_rank")

        if len(series_slices) == 0:
            continue

        n_slices = int(series_slices["n_slices"].iloc[0])

        rel_pos = RELATIVE_SLICE_CENTER_RULES[task].get(
            level,
            RELATIVE_SLICE_CENTER_RULES[task]["all"],
        )

        rank = rel_to_rank(rel_pos, n_slices)
        center = series_slices[series_slices["slice_rank"].eq(rank)].iloc[0]

        out = row.to_dict()
        out.update({
            "selector_task": task,
            "selected_center_rank": rank,
            "selected_center_rel_pos": rel_pos,
            "selected_img_path_center": center["img_path"],
            "selected_n_slices": n_slices,
        })

        selected.append(out)

    return pd.DataFrame(selected)

In [ ]:
def pixel_to_patient_3d(row, x, y):
    origin = np.array([row["ipp_x"], row["ipp_y"], row["ipp_z"]], dtype=float)
    row_dir = np.array([row["row_x"], row["row_y"], row["row_z"]], dtype=float)
    col_dir = np.array([row["col_x"], row["col_y"], row["col_z"]], dtype=float)

    ps_row = float(row["pixel_spacing_0"])
    ps_col = float(row["pixel_spacing_1"])

    return origin + x * ps_col * row_dir + y * ps_row * col_dir

In [ ]:
def select_axial_slices_from_vnet(study_df, study_geom, vnet_points_df):
    axial_rows = study_df[
        study_df["plane"].eq("Axial")
        & study_df["level"].isin(LEVEL_ORDER)
    ].copy()

    axial_geom = study_geom[study_geom["plane"].eq("Axial")].copy()

    axial_rows = axial_rows.merge(
        vnet_points_df,
        on=["study_id", "level"],
        how="inner",
    )

    selected = []

    for _, row in axial_rows.iterrows():
        series_id = str(row["series_id"])

        series_slices = axial_geom[
            axial_geom["series_id"].astype(str).eq(series_id)
        ].copy()

        if len(series_slices) == 0:
            continue

        point = np.array([row["pt_x"], row["pt_y"], row["pt_z"]], dtype=float)

        origins = series_slices[["ipp_x", "ipp_y", "ipp_z"]].values.astype(float)
        normals = series_slices[["normal_x", "normal_y", "normal_z"]].values.astype(float)
        normals = normals / np.linalg.norm(normals, axis=1, keepdims=True)

        distances = np.abs(np.sum((point - origins) * normals, axis=1))
        nearest_idx = int(np.argmin(distances))

        center = series_slices.iloc[nearest_idx]

        out = row.to_dict()
        out.update({
            "selected_center_rank": int(center["slice_rank"]),
            "selected_img_path_center": center["img_path"],
            "selected_n_slices": int(center["n_slices"]),
            "nearest_plane_distance_mm": float(distances[nearest_idx]),
        })

        selected.append(out)

    return pd.DataFrame(selected)

In [ ]:
import numpy as np

import matplotlib.pyplot as plt

from matplotlib.patches import Patch

BASE = "#d9d9d9"

COLORS = {
    "spinal": ("#4c78a8", "#a9c2df"),
    "foraminal": ("#f28e2b", "#f8c58d"),
    "axial": ("#59a14f", "#a8d5a2"),
}

def selection_type(condition):
    c = str(condition).lower()
    if "spinal canal" in c:
        return "spinal"
    if "foraminal" in c:
        return "foraminal"
    if "subarticular" in c:
        return "axial"
    return "other"

def map_rank(rank, n_slices, common_n):
    if n_slices <= 1 or common_n <= 1:
        return 1

    rel = (int(rank) - 1) / (int(n_slices) - 1)
    return int(round(rel * (common_n - 1))) + 1

def make_level_colors(df_level, common_n, default_type):
    colors = [BASE] * common_n

    # neighbours first
    for _, row in df_level.iterrows():
        t = selection_type(row["condition"])
        if t == "other":
            t = default_type

        center = map_rank(
            row["selected_center_rank"],
            row["selected_n_slices"],
            common_n,
        )

        _, neighbor_color = COLORS[t]

        for r in [center - 1, center + 1]:
            if 1 <= r <= common_n and colors[r - 1] == BASE:
                colors[r - 1] = neighbor_color

    # centers second, so they overwrite neighbours
    for _, row in df_level.iterrows():
        t = selection_type(row["condition"])
        if t == "other":
            t = default_type

        center = map_rank(
            row["selected_center_rank"],
            row["selected_n_slices"],
            common_n,
        )

        center_color, _ = COLORS[t]

        if 1 <= center <= common_n:
            colors[center - 1] = center_color

    return colors

def plot_slice_selection_panel(df, title, default_type):
    levels = [lvl for lvl in LEVEL_ORDER if lvl in set(df["level"])]

    fig, axes = plt.subplots(
        len(levels), 1,
        figsize=(12, 1.0 * len(levels) + 1.5),
        squeeze=False
    )
    axes = axes.flatten()

    for ax, level in zip(axes, levels):
        d = df[df["level"].eq(level)].copy()

        common_n = int(d["selected_n_slices"].max())
        colors = make_level_colors(d, common_n, default_type)

        x = np.arange(1, common_n + 1)

        ax.bar(
            x,
            np.ones(common_n),
            color=colors,
            edgecolor="black",
            linewidth=0.7,
            width=0.85,
        )

        ax.set_xlim(0.5, common_n + 0.5)
        ax.set_ylim(0, 1.05)
        ax.set_yticks([])
        ax.set_ylabel(level, rotation=0, ha="right", va="center", fontsize=10)
        ax.grid(axis="x", alpha=0.25)

        if common_n > 25:
            ax.set_xticks(np.arange(1, common_n + 1, 2))
        else:
            ax.set_xticks(np.arange(1, common_n + 1))

        for spine in ["top", "right", "left"]:
            ax.spines[spine].set_visible(False)

    axes[-1].set_xlabel("Slice rank")
    fig.suptitle(title, x=0.01, ha="left", fontsize=13)

    legend = [
        Patch(facecolor=BASE, edgecolor="black", label="Other slices"),
        Patch(facecolor=COLORS["spinal"][0], edgecolor="black", label="Spinal center"),
        Patch(facecolor=COLORS["spinal"][1], edgecolor="black", label="Spinal ±1"),
        Patch(facecolor=COLORS["foraminal"][0], edgecolor="black", label="Foraminal center"),
        Patch(facecolor=COLORS["foraminal"][1], edgecolor="black", label="Foraminal ±1"),
        Patch(facecolor=COLORS["axial"][0], edgecolor="black", label="Axial center"),
        Patch(facecolor=COLORS["axial"][1], edgecolor="black", label="Axial ±1"),
    ]

    fig.legend(
        handles=legend,
        loc="upper center",
        bbox_to_anchor=(0.5, 1.02),
        ncol=4,
        frameon=False,
    )

    plt.tight_layout()
    plt.show()

In [ ]:
IMG_SIZE = 256

NUM_SLICES = 3

BASE_CHANNELS = 32

DROPOUT = 0.0

LEVEL_ORDER = ["L1/L2", "L2/L3", "L3/L4", "L4/L5", "L5/S1"]

SIDES = ["Left", "Right"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def load_unet(ckpt_path, n_targets):
    model = get_localization_model(
        model_name="unet",
        in_channels=NUM_SLICES,
        out_channels=1,
        n_levels=n_targets,
        base_c=BASE_CHANNELS,
        dropout=DROPOUT,
        img_size=IMG_SIZE,
    ).to(device)

    ckpt = torch.load(ckpt_path, map_location=device)
    state = ckpt["model_state_dict"] if "model_state_dict" in ckpt else ckpt

    model.load_state_dict(state)
    model.eval()
    return model

In [ ]:
def read_dicom_2d(path):
    ds = pydicom.dcmread(str(path))
    img = ds.pixel_array.astype(np.float32)

    if getattr(ds, "PhotometricInterpretation", "") == "MONOCHROME1":
        img = img.max() - img

    lo, hi = np.percentile(img, [1, 99])
    img = np.clip(img, lo, hi)
    img = (img - lo) / (hi - lo + 1e-6)

    return img.astype(np.float32), ds

def resize_2d(img, size=IMG_SIZE):
    x = torch.from_numpy(img)[None, None].float()
    x = F.interpolate(x, size=(size, size), mode="bilinear", align_corners=False)
    return x[0, 0].numpy().astype(np.float32)

def load_25d_stack(series_geom, center_rank, img_size=IMG_SIZE):
    series_geom = series_geom.sort_values("slice_rank").reset_index(drop=True)

    n = len(series_geom)
    center_rank = int(center_rank)

    ranks = [
        max(1, center_rank - 1),
        center_rank,
        min(n, center_rank + 1),
    ]

    imgs = []
    center_ds = None

    for r in ranks:
        row = series_geom[series_geom["slice_rank"].eq(r)].iloc[0]
        img, ds = read_dicom_2d(row["img_path"])

        if r == center_rank:
            center_ds = ds

        imgs.append(resize_2d(img, img_size))

    stack = np.stack(imgs).astype(np.float32)
    stack = (stack - stack.mean()) / (stack.std() + 1e-6)

    return stack, center_ds

In [ ]:
def read_dicom(path):
    ds = pydicom.dcmread(str(path))
    img = ds.pixel_array.astype(np.float32)

    if getattr(ds, "PhotometricInterpretation", "") == "MONOCHROME1":
        img = img.max() - img

    lo, hi = np.percentile(img, [1, 99])
    img = np.clip(img, lo, hi)
    img = (img - lo) / (hi - lo + 1e-6)

    return img.astype(np.float32), ds

def resize_img(img, size=IMG_SIZE):
    x = torch.from_numpy(img)[None, None].float()
    x = F.interpolate(x, size=(size, size), mode="bilinear", align_corners=False)
    return x[0, 0].numpy()

def load_25d_stack(series_geom, center_rank):
    series_geom = series_geom.sort_values("slice_rank").reset_index(drop=True)

    n = len(series_geom)
    center_rank = int(center_rank)

    ranks = [
        max(1, center_rank - 1),
        center_rank,
        min(n, center_rank + 1),
    ]

    imgs = []
    center_ds = None
    center_path = None

    for r in ranks:
        row = series_geom[series_geom["slice_rank"].eq(r)].iloc[0]
        img, ds = read_dicom(row["img_path"])

        if r == center_rank:
            center_ds = ds
            center_path = row["img_path"]

        imgs.append(resize_img(img))

    stack = np.stack(imgs).astype(np.float32)
    stack = (stack - stack.mean()) / (stack.std() + 1e-6)

    return stack, center_ds, center_path

def heatmap_argmax(hmap):
    y, x = np.unravel_index(np.argmax(hmap), hmap.shape)
    return x, y, hmap[y, x]

In [ ]:
@torch.no_grad()
def predict_raw_unet(model, stack, targets):
    model.eval()

    image = torch.from_numpy(stack)[None].float().to(device)

    rows = []

    for target_idx, target in enumerate(targets):
        target_tensor = torch.tensor([target_idx], dtype=torch.long).to(device)

        logits = model(image, target_tensor)
        hmap = torch.sigmoid(logits)[0, 0].cpu().numpy()

        x, y, conf = heatmap_argmax(hmap)

        rows.append({
            "target": target,
            "target_idx": target_idx,
            "pred_x_resized": x,
            "pred_y_resized": y,
            "raw_confidence": conf,
        })

    return pd.DataFrame(rows)

In [ ]:
import torch

import torch.nn.functional as F

import matplotlib.pyplot as plt

import numpy as np

In [ ]:
def resize_map(arr, out_h, out_w):
    x = torch.from_numpy(arr)[None, None].float()
    x = F.interpolate(x, size=(out_h, out_w), mode="bilinear", align_corners=False)
    return x[0, 0].numpy()

In [ ]:
@torch.no_grad()
def predict_unet_with_prior(model, stack, targets, priors):
    model.eval()

    image = torch.from_numpy(stack)[None].float().to(device)
    rows = []

    for target_idx, target in enumerate(targets):
        target_tensor = torch.tensor([target_idx], dtype=torch.long).to(device)

        logits = model(image, target_tensor)
        raw_hmap = torch.sigmoid(logits)[0, 0].cpu().numpy()

        fused_hmap = fuse_with_prior(
            raw_hmap=raw_hmap,
            prior_hmap=priors[target_idx],
        )

        x, y, conf = argmax_xy(fused_hmap)

        rows.append({
            "target": target,
            "target_idx": target_idx,
            "pred_x_resized": x,
            "pred_y_resized": y,
            "corrected_confidence": conf,
        })

    return pd.DataFrame(rows)

In [ ]:
def plot_sagittal_raw_vs_corrected(
    sag_img_path,
    sag_raw_pred_df,
    sag_corrected_df,
    SAG_PRIORS,
):
    img, _ = read_dicom(sag_img_path)
    h, w = img.shape

    fig, axes = plt.subplots(len(LEVEL_ORDER), 2, figsize=(10, 3 * len(LEVEL_ORDER)))

    for i, level in enumerate(LEVEL_ORDER):
        raw_row = sag_raw_pred_df[sag_raw_pred_df["target"].eq(level)].iloc[0]
        corr_row = sag_corrected_df[sag_corrected_df["target"].eq(level)].iloc[0]

        prior = resize_map(SAG_PRIORS[i], h, w)

        # Left: raw + prior
        ax = axes[i, 0]
        ax.imshow(img, cmap="gray")
        ax.imshow(prior, cmap="jet", alpha=0.25)
        ax.scatter(raw_row["pred_x"], raw_row["pred_y"], c="lime", s=50)
        ax.set_title(f"{level} | Raw + prior")
        ax.axis("off")

        # Right: corrected + prior
        ax = axes[i, 1]
        ax.imshow(img, cmap="gray")
        ax.imshow(prior, cmap="jet", alpha=0.25)
        ax.scatter(corr_row["pred_x"], corr_row["pred_y"], c="red", s=50)
        ax.set_title(f"{level} | Corrected + prior")
        ax.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
def plot_axial_raw_vs_corrected(
    ax_raw_pred_df,
    ax_corrected_df,
    AX_PRIORS,
):
    for (series_id, level, slice_rank), raw_group in ax_raw_pred_df.groupby(["series_id", "level", "slice_rank"]):
        corr_group = ax_corrected_df[
            ax_corrected_df["series_id"].eq(series_id)
            & ax_corrected_df["level"].eq(level)
            & ax_corrected_df["slice_rank"].eq(slice_rank)
        ].copy()

        img_path = raw_group["img_path"].iloc[0]
        img, _ = read_dicom(img_path)
        h, w = img.shape

        fig, axes = plt.subplots(len(SIDES), 2, figsize=(9, 4 * len(SIDES)))

        for i, side in enumerate(SIDES):
            raw_row = raw_group[raw_group["target"].eq(side)].iloc[0]
            corr_row = corr_group[corr_group["target"].eq(side)].iloc[0]

            prior = resize_map(AX_PRIORS[i], h, w)

            # Left: raw + prior
            ax = axes[i, 0]
            ax.imshow(img, cmap="gray")
            ax.imshow(prior, cmap="jet", alpha=0.25)
            ax.scatter(raw_row["pred_x"], raw_row["pred_y"], c="lime", s=50)
            ax.set_title(f"{level} | {side} | Raw + prior")
            ax.axis("off")

            # Right: corrected + prior
            ax = axes[i, 1]
            ax.imshow(img, cmap="gray")
            ax.imshow(prior, cmap="jet", alpha=0.25)
            ax.scatter(corr_row["pred_x"], corr_row["pred_y"], c="red", s=50)
            ax.set_title(f"{level} | {side} | Corrected + prior")
            ax.axis("off")

        plt.tight_layout()
        plt.show()

In [ ]:
import numpy as np

import pandas as pd

import torch

import torch.nn.functional as F

import matplotlib.pyplot as plt

import pydicom

import numpy as np

import torch

import torch.nn.functional as F

import matplotlib.pyplot as plt

import matplotlib.patches as patches

In [ ]:
def read_img(path):
    ds = pydicom.dcmread(str(path))
    img = ds.pixel_array.astype(np.float32)

    if getattr(ds, "PhotometricInterpretation", "") == "MONOCHROME1":
        img = img.max() - img

    lo, hi = np.percentile(img, [1, 99])
    img = np.clip(img, lo, hi)
    img = (img - lo) / (hi - lo + 1e-6)

    return img.astype(np.float32)

def crop_square(img, x, y, size=ROI_SIZE):
    h, w = img.shape
    half = size // 2

    x = int(round(x))
    y = int(round(y))

    x1, x2 = x - half, x + half
    y1, y2 = y - half, y + half

    crop = np.zeros((size, size), dtype=np.float32)

    sx1 = max(0, x1)
    sx2 = min(w, x2)
    sy1 = max(0, y1)
    sy2 = min(h, y2)

    dx1 = sx1 - x1
    dx2 = dx1 + (sx2 - sx1)
    dy1 = sy1 - y1
    dy2 = dy1 + (sy2 - sy1)

    crop[dy1:dy2, dx1:dx2] = img[sy1:sy2, sx1:sx2]

    return crop

def load_roi_25d(series_geom, center_rank, x, y, crop_size=ROI_SIZE):
    series_geom = series_geom.sort_values("slice_rank").reset_index(drop=True)

    n = int(series_geom["slice_rank"].max())
    center_rank = int(center_rank)

    ranks = [
        max(1, center_rank - 1),
        center_rank,
        min(n, center_rank + 1),
    ]

    crops = []

    for r in ranks:
        img_path = series_geom[series_geom["slice_rank"].eq(r)]["img_path"].iloc[0]
        img = read_img(img_path)
        crop = crop_square(img, x, y, size=crop_size)
        crops.append(crop)

    stack = np.stack(crops).astype(np.float32)  # [3, 96, 96]

    return stack

In [ ]:
CLS_IMG_SIZE = 224

def preprocess_roi_for_classification(stack, out_size=CLS_IMG_SIZE):
    """
    Input:  stack [3, H, W]
    Output: tensor [3, out_size, out_size]
    """
    x = torch.from_numpy(stack).float().unsqueeze(0)  # [1, 3, H, W]
    x = F.interpolate(x, size=(out_size, out_size), mode="bilinear", align_corners=False)
    return x[0]  # [3, out_size, out_size]

In [ ]:
def show_classification_roi_pipeline(cls_roi_df, n=3):
    d = cls_roi_df.head(n).reset_index(drop=True)

    fig, axes = plt.subplots(
        len(d),
        3,
        figsize=(15, 4.5 * len(d)),
    )

    if len(d) == 1:
        axes = np.expand_dims(axes, axis=0)

    for i, row in d.iterrows():
        stack = row["roi_stack"]                  # [3, 96, 96]
        tensor = preprocess_roi_for_classification(stack)  # [3, 224, 224]

        img = read_img(row["img_path"])

        title_side = row["side"] if pd.notna(row["side"]) else "center"
        title = f"{row['condition']} | {row['level']} | {title_side}"

        # ---------------------------------------------------------
        # 1) Original image + ROI box
        # ---------------------------------------------------------
        ax = axes[i, 0]
        ax.imshow(img, cmap="gray")

        x = float(row["x"])
        y = float(row["y"])
        half = ROI_SIZE // 2

        rect = patches.Rectangle(
            (x - half, y - half),
            ROI_SIZE,
            ROI_SIZE,
            linewidth=2,
            edgecolor="red",
            facecolor="none",
        )

        ax.add_patch(rect)
        ax.scatter(x, y, c="yellow", s=35)
        ax.set_title(f"Original + ROI\n{title}", fontsize=10)
        ax.axis("off")

        # ---------------------------------------------------------
        # 2) Raw model image as RGB-like 2.5D stack
        # ---------------------------------------------------------
        raw_rgb = np.transpose(stack, (1, 2, 0))  # [H, W, 3]
        raw_rgb = (raw_rgb - raw_rgb.min()) / (raw_rgb.max() - raw_rgb.min() + 1e-8)

        ax = axes[i, 1]
        ax.imshow(raw_rgb)
        ax.set_title(f"Raw model image\nshape={raw_rgb.shape}", fontsize=10)
        ax.axis("off")

        # ---------------------------------------------------------
        # 3) Final channels after classification preprocessing
        # ---------------------------------------------------------
        final = tensor.numpy()  # [3, 224, 224]

        concat = np.concatenate(
            [final[0], final[1], final[2]],
            axis=1,
        )

        ax = axes[i, 2]
        ax.imshow(concat, cmap="gray")
        ax.set_title(
            "Final channels: ['prev', 'current', 'next']\n"
            f"tensor={tuple(final.shape)}",
            fontsize=10,
        )
        ax.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
from pathlib import Path

import os

import gc

import warnings

import numpy as np

import pandas as pd

import torch

import matplotlib.pyplot as plt

from torch.utils.data import DataLoader

from torchvision import transforms

from sklearn.model_selection import GroupShuffleSplit

from sklearn.metrics import (
    log_loss,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    RocCurveDisplay,
)

from sklearn.linear_model import LogisticRegression

from sklearn.tree import DecisionTreeClassifier, plot_tree

In [ ]:
def get_model_specs(condition_key):
    return {
        "resnet34": {
            "backbone": "resnet34",
            "checkpoint": MODEL_ROOT / "models_classification_ResNet34_96_zoom_resnet34" / f"{condition_key}_best.pt",
        },
        "efficientnet_b0": {
            "backbone": "efficientnet_b0",
            "checkpoint": MODEL_ROOT / "models_classification_EfficientNet_b_96_zoom_efficientnet_b0" / f"{condition_key}_best.pt",
        },
        "densenet169": {
            "backbone": "densenet169",
            "checkpoint": MODEL_ROOT / "models_classification_densenet169_densenet169" / f"{condition_key}_best.pt",
        },
        "convnext_base": {
            "backbone": "convnext_base",
            "checkpoint": MODEL_ROOT / "models_classification_ConvNext_base_convnext_base" / f"{condition_key}_best.pt",
        },
        "swin_b": {
            "backbone": "swin_b",
            "checkpoint": MODEL_ROOT / "models_classification_Swin_b_96_zoom_swin_b" / f"{condition_key}_best.pt",
        },
    }

In [ ]:
def condition_to_key(condition):
    c = str(condition).lower()

    if "spinal canal" in c:
        return "spinal_canal_stenosis"
    if "foraminal" in c:
        return "neural_foraminal_narrowing"
    if "subarticular" in c:
        return "subarticular_stenosis"

    return None

def get_side(row):
    if pd.notna(row.get("side")):
        return str(row["side"]).lower()

    c = str(row["condition"]).lower()
    if "left" in c:
        return "left"
    if "right" in c:
        return "right"

    return "center"

def get_series_id_for_model(series_description):
    s = str(series_description)

    if "Sagittal T2/STIR" in s:
        return SERIES_TO_ID["Sagittal T2/STIR"]
    if "Sagittal T1" in s:
        return SERIES_TO_ID["Sagittal T1"]
    if "Axial T2" in s:
        return SERIES_TO_ID["Axial T2"]

    return 0

In [ ]:
def roi_to_tensor(stack):
    x = torch.from_numpy(stack).float().unsqueeze(0)  # [1, 3, 96, 96]
    x = F.interpolate(
        x,
        size=(IMAGE_SIZE, IMAGE_SIZE),
        mode="bilinear",
        align_corners=False,
    )
    x = (x - x.mean()) / (x.std() + 1e-6)
    return x.to(DEVICE)

In [ ]:
@torch.no_grad()
def predict_single_roi(model, row):
    image = roi_to_tensor(row["roi_stack"])

    metadata = {
        "level": torch.tensor([int(row["level_id"])], dtype=torch.long, device=DEVICE),
        "side": torch.tensor([int(row["side_id"])], dtype=torch.long, device=DEVICE),
        "series": torch.tensor([int(row["series_meta_id"])], dtype=torch.long, device=DEVICE),
    }

    logits = model(image, metadata)

    probs = torch.softmax(logits, dim=1)[0].detach().cpu().numpy()
    pred_idx = int(probs.argmax())

    return {
        "p0": float(probs[0]),
        "p1": float(probs[1]),
        "p2": float(probs[2]),
        "pred_idx": pred_idx,
        "pred_class": TARGET_NAMES[pred_idx],
        "confidence": float(probs[pred_idx]),
    }

In [ ]:
import numpy as np

import pandas as pd

import torch

import torch.nn.functional as F

import joblib

import gc

from pathlib import Path

In [ ]:
from scipy.ndimage import gaussian_filter
import numpy as np

def norm01(x, eps=1e-8):
    x = np.asarray(x, dtype=np.float32)
    return (x - x.min()) / (x.max() - x.min() + eps)


def fuse_with_prior(raw_hmap, prior_hmap):
    raw = norm01(raw_hmap)

    prior = norm01(prior_hmap)
    prior = gaussian_filter(
        prior,
        sigma=(PRIOR_SIGMA_Y, PRIOR_SIGMA_X),
    )
    prior = norm01(prior)

    fused = np.log(raw + 1e-8) + PRIOR_LAMBDA * np.log(prior + 1e-8)

    return fused


def argmax_xy(hmap):
    y, x = np.unravel_index(np.argmax(hmap), hmap.shape)
    return float(x), float(y), float(hmap[y, x])

In [ ]:
def read_img_simple(path):
    try:
        return read_img(path)
    except NameError:
        ds = pydicom.dcmread(str(path))
        img = ds.pixel_array.astype(np.float32)

        if getattr(ds, "PhotometricInterpretation", "") == "MONOCHROME1":
            img = img.max() - img

        lo, hi = np.percentile(img, [1, 99])
        img = np.clip(img, lo, hi)
        img = (img - lo) / (hi - lo + 1e-6)

        return img.astype(np.float32)


def draw_point_and_box(ax, x, y, label, color="yellow", box_color="red", box=True):
    ax.scatter(x, y, c=color, s=35)

    ax.text(
        x + 5,
        y,
        str(label),
        color=color,
        fontsize=9,
        va="center",
        bbox=dict(facecolor="black", alpha=0.4, edgecolor="none", pad=1),
    )

    if box:
        half = ROI_SIZE // 2
        rect = patches.Rectangle(
            (x - half, y - half),
            ROI_SIZE,
            ROI_SIZE,
            linewidth=1.8,
            edgecolor=box_color,
            facecolor="none",
        )
        ax.add_patch(rect)

In [ ]:
def plot_sagittal_raw_vs_corrected_with_rois(
    study_geom,
    vnet_level_df,
    sag_raw_pred_df,
    sag_corrected_df,
    cls_roi_df,
):
    sag_series_id = vnet_level_df["series_id"].iloc[0]

    sag_geom = study_geom[
        study_geom["series_id"].astype(str).eq(str(sag_series_id))
    ].sort_values("slice_rank")

    center_rank = int(round(sag_geom["n_slices"].iloc[0] / 2))

    img_path = sag_geom[
        sag_geom["slice_rank"].eq(center_rank)
    ]["img_path"].iloc[0]

    img = read_img_simple(img_path)

    fig, axes = plt.subplots(1, 2, figsize=(13, 8))

    # Raw
    axes[0].imshow(img, cmap="gray")
    axes[0].set_title("Raw sagittal level predictions")
    axes[0].axis("off")

    for _, row in sag_raw_pred_df.iterrows():
        draw_point_and_box(
            axes[0],
            row["pred_x"],
            row["pred_y"],
            row["target"],
            color="lime",
            box_color="lime",
            box=False,
        )

    # Corrected
    axes[1].imshow(img, cmap="gray")
    axes[1].set_title("Corrected sagittal predictions + ROI boxes")
    axes[1].axis("off")

    for _, row in sag_corrected_df.iterrows():
        draw_point_and_box(
            axes[1],
            row["pred_x"],
            row["pred_y"],
            row["target"],
            color="yellow",
            box_color="red",
            box=False,
        )

    sag_rois = cls_roi_df[cls_roi_df["plane"].eq("Sagittal")].copy()

    for _, row in sag_rois.iterrows():
        draw_point_and_box(
            axes[1],
            row["x"],
            row["y"],
            row["level"],
            color="yellow",
            box_color="red",
            box=True,
        )

    plt.tight_layout()
    plt.show()

In [ ]:
def plot_axial_raw_vs_corrected_with_rois(
    ax_raw_pred_df,
    ax_corrected_df,
    cls_roi_df,
):
    levels = [lvl for lvl in LEVEL_ORDER if lvl in set(ax_corrected_df["level"])]

    fig, axes = plt.subplots(
        len(levels),
        2,
        figsize=(12, 5 * len(levels)),
        squeeze=False,
    )

    for i, level in enumerate(levels):
        corr_level = ax_corrected_df[ax_corrected_df["level"].eq(level)].copy()
        raw_level = ax_raw_pred_df[ax_raw_pred_df["level"].eq(level)].copy()

        if len(corr_level) == 0:
            continue

        img_path = corr_level["img_path"].iloc[0]
        img = read_img_simple(img_path)

        # Raw
        ax = axes[i, 0]
        ax.imshow(img, cmap="gray")
        ax.set_title(f"{level} | raw axial predictions")
        ax.axis("off")

        for _, row in raw_level.iterrows():
            color = "cyan" if row["target"] == "Left" else "lime"
            draw_point_and_box(
                ax,
                row["pred_x"],
                row["pred_y"],
                row["target"],
                color=color,
                box_color=color,
                box=False,
            )

        # Corrected + ROI boxes
        ax = axes[i, 1]
        ax.imshow(img, cmap="gray")
        ax.set_title(f"{level} | corrected axial predictions + ROI boxes")
        ax.axis("off")

        for _, row in corr_level.iterrows():
            color = "cyan" if row["target"] == "Left" else "lime"
            draw_point_and_box(
                ax,
                row["pred_x"],
                row["pred_y"],
                row["target"],
                color=color,
                box_color="red",
                box=False,
            )

        roi_level = cls_roi_df[
            cls_roi_df["plane"].eq("Axial")
            & cls_roi_df["level"].eq(level)
        ].copy()

        for _, row in roi_level.iterrows():
            label = row["side"] if pd.notna(row["side"]) else ""
            draw_point_and_box(
                ax,
                row["x"],
                row["y"],
                label,
                color="yellow",
                box_color="red",
                box=True,
            )

    plt.tight_layout()
    plt.show()

In [ ]:
def make_condition_prediction_summary(model_pred_df):
    model_table = model_pred_df.pivot_table(
        index=["roi_idx", "condition", "level", "side"],
        columns="model",
        values="pred_class",
        aggfunc="first",
    ).reset_index()

    prob_table = (
        model_pred_df
        .groupby(["roi_idx", "condition", "level", "side"], dropna=False)[["p0", "p1", "p2"]]
        .mean()
        .reset_index()
        .rename(columns={
            "p0": "avg_p_normal_mild",
            "p1": "avg_p_moderate",
            "p2": "avg_p_severe",
        })
    )

    out = model_table.merge(
        prob_table,
        on=["roi_idx", "condition", "level", "side"],
        how="left",
    )

    probs = out[["avg_p_normal_mild", "avg_p_moderate", "avg_p_severe"]].values
    out["mean_prediction"] = [TARGET_NAMES[i] for i in probs.argmax(axis=1)]

    return out


def color_prediction_cells(value):
    if value == "Normal/Mild":
        return "background-color: #b7e4c7"
    if value == "Moderate":
        return "background-color: #fff3b0"
    if value == "Severe":
        return "background-color: #f4a6a6"
    return ""


def display_condition_prediction_tables(model_pred_df):
    summary = make_condition_prediction_summary(model_pred_df)

    model_cols = [
        c for c in summary.columns
        if c not in [
            "roi_idx",
            "condition",
            "level",
            "side",
            "avg_p_normal_mild",
            "avg_p_moderate",
            "avg_p_severe",
            "mean_prediction",
        ]
    ]

    display_cols = [
        "condition",
        "level",
        "side",
    ] + model_cols + [
        "mean_prediction",
        "avg_p_normal_mild",
        "avg_p_moderate",
        "avg_p_severe",
    ]

    for condition in summary["condition"].dropna().unique():
        display(Markdown(f"### {condition}"))

        d = (
            summary[summary["condition"].eq(condition)]
            [display_cols]
            .sort_values(["level", "side"])
        )

        styled = (
            d.style
            .applymap(color_prediction_cells, subset=model_cols + ["mean_prediction"])
            .format({
                "avg_p_normal_mild": "{:.3f}",
                "avg_p_moderate": "{:.3f}",
                "avg_p_severe": "{:.3f}",
            })
        )

        display(styled)

In [ ]:
def display_final_ensemble_table(final_ensemble_df):
    final_cols = [
        "condition",
        "level",
        "side",
        "mean_pred_class",
        "max_model_pred_class",
        "final_pred_class",
        "mean_confidence",
    ]

    final_cols = [c for c in final_cols if c in final_ensemble_df.columns]

    d = final_ensemble_df[final_cols].copy()

    display(
        d.sort_values(["condition", "level", "side"])
        .style
        .applymap(
            color_prediction_cells,
            subset=["mean_pred_class", "max_model_pred_class", "final_pred_class"],
        )
        .format({"mean_confidence": "{:.3f}"})
    )

In [ ]:
def plot_vnet_level_localization(study_geom, vnet_level_df):
    sag_series_id = str(vnet_level_df["series_id"].iloc[0])

    sag_geom = study_geom[
        study_geom["series_id"].astype(str).eq(sag_series_id)
    ].sort_values("slice_rank")

    center_rank = int(round(sag_geom["n_slices"].iloc[0] / 2))
    img_path = sag_geom[sag_geom["slice_rank"].eq(center_rank)]["img_path"].iloc[0]

    img = read_img_simple(img_path)

    plt.figure(figsize=(5, 7))
    plt.imshow(img, cmap="gray")

    for _, row in vnet_level_df.iterrows():
        y = row["pred_y_original"] if "pred_y_original" in row else row["pred_y"]

        plt.axhline(y, linewidth=1.5)
        plt.text(
            5,
            y,
            row["level"],
            color="yellow",
            fontsize=10,
            va="center",
            bbox=dict(facecolor="black", alpha=0.4, edgecolor="none", pad=1),
        )

    plt.title("V-Net predicted lumbar level heights")
    plt.axis("off")
    plt.show()

In [ ]:
def display_visual_medical_report_from_pipeline(
    study_id,
    study_geom,
    vnet_level_df,
    selected_slices_df,
    sag_raw_pred_df,
    sag_corrected_df,
    ax_raw_pred_df,
    ax_corrected_df,
    cls_roi_df,
    model_pred_df,
    final_ensemble_df=None,
):
    display(Markdown(f"# Explainability Report — Study `{study_id}`"))

    display(Markdown("## 1. Series Overview"))

    try:
        n_series = study_geom["series_id"].nunique()
        n_images = study_geom["img_path"].nunique()

        display(Markdown(
            f"""
            **Number of series:** {n_series}  
            **Number of DICOM images:** {n_images}
            """
        ))

        series_counts = (
            study_geom
            .groupby(["series_description", "plane"])
            .size()
            .reset_index(name="n_slices")
            .sort_values(["plane", "series_description"])
        )

        fig, ax = plt.subplots(figsize=(5, 2))
        ax.barh(
            series_counts["series_description"] + " | " + series_counts["plane"],
            series_counts["n_slices"],
        )
        ax.set_xlabel("Number of slices")
        ax.set_title("Available series")
        plt.tight_layout()
        plt.show()

    except Exception as e:
        print("Could not show series overview:", e)

    display(Markdown("## 2. V-Net Sagittal Level Localization"))

    try:
        plot_vnet_level_localization(
            study_geom=study_geom,
            vnet_level_df=vnet_level_df,
        )
    except Exception as e:
        print("Could not plot V-Net level localization:", e)

    display(Markdown("## 3. Selected Slices"))

    try:
        sag_sel = selected_slices_df[
            selected_slices_df["selection_source"].astype(str).str.contains("sag", case=False, na=False)
        ]

        ax_sel = selected_slices_df[
            selected_slices_df["selection_source"].astype(str).str.contains("ax", case=False, na=False)
        ]

        if len(sag_sel) > 0:
            plot_slice_selection_panel(
                sag_sel,
                title="Sagittal selected slices by level",
                default_type="foraminal",
            )

        if len(ax_sel) > 0:
            plot_slice_selection_panel(
                ax_sel,
                title="Axial selected slices by level",
                default_type="axial",
            )

    except Exception as e:
        print("Could not plot selected slices:", e)

    display(Markdown("## 4. Sagittal Localization: Raw vs Corrected"))

    try:
        plot_sagittal_raw_vs_corrected_with_rois(
            study_geom=study_geom,
            vnet_level_df=vnet_level_df,
            sag_raw_pred_df=sag_raw_pred_df,
            sag_corrected_df=sag_corrected_df,
            cls_roi_df=cls_roi_df,
        )
    except Exception as e:
        print("Could not plot sagittal raw/corrected predictions:", e)

    display(Markdown("## 5. Axial Localization: Raw vs Corrected"))

    try:
        plot_axial_raw_vs_corrected_with_rois(
            ax_raw_pred_df=ax_raw_pred_df,
            ax_corrected_df=ax_corrected_df,
            cls_roi_df=cls_roi_df,
        )
    except Exception as e:
        print("Could not plot axial raw/corrected predictions:", e)

    display(Markdown("## 6. Model Predictions by Condition"))

    display_condition_prediction_tables(model_pred_df)

    if final_ensemble_df is not None:
        display(Markdown("## 7. Final Safety-Net Ensemble Summary"))
        display_final_ensemble_table(final_ensemble_df)

    display(Markdown("---"))
    display(Markdown("**Report complete.**"))